# Mock DESI DR2 generation

Creates a HATS catalog of simulated DESI DR2 data, in the spirit of
[Mock_DP1_generation](https://github.com/lsst-so/linccf/blob/main/internal/LSSTCam_init/Mock_DP1_generation.ipynb),
but generated purely from the catalog's `_metadata` file — no access to the real data is needed.

The `_metadata` file aggregates the parquet footers of every partition (one row group per
partition file), so for each partition we know its pixel, its row count, and per-column
statistics (min/max/null count).

### Data requirements

- `_healpix_29` values are sampled uniformly within each partition's pixel, and RA/DEC are derived from them;
- Numeric columns are re-generated uniformly within the partition-level min/max;
- Booleans are random; string and dictionary columns sample from the min/max anchor values;
- `COEFF` (fixed-size list of 5 floats) is sampled uniformly within its element statistics;
- Nulls are injected at each column's observed null fraction;
- Columns with `inf` in the real data (`*_EW_IVAR`) have their bounds clamped to the dtype's finite range.

The default `ROW_SCALE = 0.01` yields a small catalog (~38k rows) with the full partition
structure (238 pixels). Set `ROW_SCALE = 1.0` for full catalog scale (38M rows — very large on disk).

In [ ]:
import re

import cdshealpix
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm

from hats.catalog import PartitionInfo, TableProperties
from hats.io import paths
from hats.io.parquet_metadata import write_parquet_metadata
from hats.pixel_math import HealpixPixel
from hats.pixel_math.spatial_index import (
    SPATIAL_INDEX_COLUMN,
    healpix_to_spatial_index,
    spatial_index_to_healpix,
)

In [ ]:
metadata_file = Path("_metadata")
output_dir = Path("mock_desi_dr2")
catalog_name = "mock_desi_dr2"

ROW_SCALE = 0.01  # fraction of the real per-partition row counts
SEED = 42

### Parse partition statistics from `_metadata`

In [ ]:
metadata = pq.read_metadata(metadata_file)
schema = metadata.schema.to_arrow_schema()

partitions = []
for i in range(metadata.num_row_groups):
    row_group = metadata.row_group(i)
    path = row_group.column(0).file_path
    match = re.search(r"Norder=(\d+)/Dir=\d+/Npix=(\d+)", path)
    pixel = HealpixPixel(int(match.group(1)), int(match.group(2)))
    stats = {}
    for j in range(row_group.num_columns):
        column = row_group.column(j)
        # leaf paths like "COEFF.list.element" map back to the "COEFF" field
        name = column.path_in_schema.split(".")[0]
        stats[name] = column.statistics
    partitions.append((pixel, row_group.num_rows, stats))

print(f"{len(partitions)} partitions, {sum(n for _, n, _ in partitions):,} total rows")

### Mock data generation

In [ ]:
def mock_partition(rng, pixel, n_rows, stats):
    """Generates a partition of mock data matching the real schema and statistics."""
    index = mock_spatial_index(rng, pixel, n_rows)
    ipix29 = spatial_index_to_healpix(index)
    ra, dec = cdshealpix.healpix_to_lonlat(ipix29, depth=29)

    arrays = []
    for field in schema:
        if field.name == SPATIAL_INDEX_COLUMN:
            array = pa.array(index, type=field.type)
        elif field.name == "RA":
            array = pa.array(ra.deg, type=field.type)
        elif field.name == "DEC":
            array = pa.array(dec.deg, type=field.type)
        else:
            array = _sample_column(rng, field, stats[field.name], n_rows)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=schema)


def mock_spatial_index(rng, pixel, n_rows):
    """Random (sorted) _healpix_29 spatial index values within the pixel."""
    low = healpix_to_spatial_index(pixel.order, pixel.pixel)
    high = healpix_to_spatial_index(pixel.order, pixel.pixel + 1)
    return np.sort(rng.integers(low, high, size=n_rows, dtype=np.int64))


def _sample_column(rng, field, stat, n_rows):
    """Samples column values uniformly within the observed min/max statistics."""
    dtype = field.type
    if stat is None:
        return pa.nulls(n_rows, type=dtype)
    if pa.types.is_fixed_size_list(dtype):
        flat = _sample_flat(rng, dtype.value_type, stat, n_rows * dtype.list_size)
        return pa.FixedSizeListArray.from_arrays(flat, dtype.list_size)
    if pa.types.is_dictionary(dtype):
        return _sample_flat(rng, dtype.value_type, stat, n_rows).dictionary_encode()
    null_mask = None
    if stat.null_count:
        null_fraction = stat.null_count / (stat.null_count + stat.num_values)
        null_mask = rng.random(n_rows) < null_fraction
    return _sample_flat(rng, dtype, stat, n_rows, null_mask)


def _sample_flat(rng, dtype, stat, n_samples, null_mask=None):
    if not stat.has_min_max:  # all-null column
        return pa.nulls(n_samples, type=dtype)
    _min, _max = stat.min, stat.max
    if pa.types.is_boolean(dtype):
        values = rng.integers(0, 2, size=n_samples).astype(bool)
    elif pa.types.is_integer(dtype):
        values = rng.integers(_min, _max, size=n_samples, endpoint=True, dtype=np.int64)
    elif pa.types.is_floating(dtype):
        # some columns contain inf in the real data; clamp to the dtype's finite range
        finfo = np.finfo(np.float32 if dtype.bit_width == 32 else np.float64)
        low = max(np.float64(_min), np.float64(-finfo.max))
        high = min(np.float64(_max), np.float64(finfo.max))
        if not np.isfinite(high - low):
            low, high = low / 2, high / 2
        values = rng.uniform(low, high, size=n_samples)
    elif pa.types.is_string(dtype):
        values = rng.choice([_min, _max], size=n_samples)
    else:
        return pa.nulls(n_samples, type=dtype)
    if null_mask is not None:
        return pa.array(values, type=dtype, mask=null_mask)
    return pa.array(values, type=dtype)

### Generate and write partitions

In [ ]:
rng = np.random.default_rng(SEED)
total_rows = 0
pixels = []
for pixel, real_rows, stats in tqdm(partitions):
    n_rows = max(1, round(real_rows * ROW_SCALE))
    table = mock_partition(rng, pixel, n_rows, stats)
    file_path = paths.pixel_catalog_file(output_dir, pixel)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(table, str(file_path))
    pixels.append(pixel)
    total_rows += n_rows
print(f"wrote {total_rows:,} rows across {len(pixels)} partitions")

### Write catalog metadata

In [ ]:
PartitionInfo.from_healpix(pixels).write_to_file(catalog_path=output_dir)
TableProperties(
    catalog_name=catalog_name,
    catalog_type="object",
    total_rows=total_rows,
    ra_column="RA",
    dec_column="DEC",
    healpix_column=SPATIAL_INDEX_COLUMN,
    healpix_order=29,
    npix_suffix=".parquet",
).to_properties_file(output_dir)
write_parquet_metadata(output_dir)

### Verify

In [ ]:
import lsdb

catalog = lsdb.open_catalog(output_dir)
catalog

In [ ]:
catalog.head()

In [ ]:
catalog.plot_pixels()

In [ ]:
catalog.cone_search(
    ra=150.11917,
    dec=2.20583,
    radius_arcsec=3600,
).write_catalog("mock_desi_dr2")

In [ ]:
!du -h

In [ ]:
from hats_import import CollectionArguments, pipeline_with_client

args = (
    CollectionArguments(
        output_artifact_name="mock_desi_dr2",
        new_catalog_name="mock_desi_dr2",
        output_path="./",
        simple_progress_bar=True,
        dask_n_workers=4,
        dask_threads_per_worker=1,
    )
    .catalog(
        catalog_path="./mock_desi_dr2/mock_desi_dr2",
    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [ ]:
from distributed import Client
from hats_import import pipeline_with_client
with Client(n_workers=4) as client:
    pipeline_with_client(args, client)